In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

In [2]:
# Section 1 — Load data
import numpy as np
import pandas as pd
import json
from sklearn.metrics import f1_score
from utils.image_config import (
    IMAGE_MANIFEST_PATH, IMAGE_EMBEDDINGS_PATH, IMAGE_CLASS_NAMES,
    IMAGE_NUM_CLASSES, IMAGE_SEED, RESULTS_DIR
)
from utils.image_embeddings import get_clip_text_embeddings
from utils.image_weak_supervision import tune_thresholds, build_clip_lfs

manifest = pd.read_csv(IMAGE_MANIFEST_PATH)
embeddings = np.load(IMAGE_EMBEDDINGS_PATH)

seed_mask = manifest["is_seed"].to_numpy().astype(bool)
seed_embs = embeddings[seed_mask]
seed_labels = manifest.loc[seed_mask, "label_idx"].to_numpy()
unlabeled_embs = embeddings[~seed_mask]

In [3]:
# Section 2 — Tune thresholds on seed, build LFs
class_prompts = [f"a photo of a {cls}" for cls in IMAGE_CLASS_NAMES]
text_embs = get_clip_text_embeddings(class_prompts)   # (10, 512) L2-normalised

thresholds = tune_thresholds(seed_embs, seed_labels, IMAGE_NUM_CLASSES, text_embs)
print("Per-class thresholds:", {IMAGE_CLASS_NAMES[i]: f"{t:.3f}" for i, t in enumerate(thresholds)})

lfs = build_clip_lfs(embeddings, IMAGE_CLASS_NAMES, text_embs, thresholds)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Per-class thresholds: {'butterfly': '0.288', 'cat': '0.269', 'chicken': '0.287', 'cow': '0.281', 'dog': '0.251', 'elephant': '0.287', 'horse': '0.267', 'sheep': '0.284', 'spider': '0.294', 'squirrel': '0.291'}


In [4]:
# Section 3 — Apply LFs to full dataset, run Snorkel LabelModel
from snorkel.labeling import LabelingFunction, PandasLFApplier
from snorkel.labeling.model import LabelModel

# Wrap callables as Snorkel LabelingFunctions
def _make_snorkel_lf(name, fn):
    return LabelingFunction(name=name, f=lambda x, _fn=fn: _fn(x.row_idx))

snorkel_lfs = [_make_snorkel_lf(f"lf_{IMAGE_CLASS_NAMES[i]}", lfs[i]) for i in range(IMAGE_NUM_CLASSES)]

# Apply to full dataset (add row_idx column for LF access)
df_for_lf = manifest.reset_index().rename(columns={"index": "row_idx"})
applier = PandasLFApplier(lfs=snorkel_lfs)
L = applier.apply(df_for_lf)
print(f"LF matrix shape: {L.shape}")  # (5500, 10)
coverage = (L != -1).any(axis=1).mean()
print(f"Coverage (at least one LF fires): {coverage:.1%}")

  0%|          | 0/5500 [00:00<?, ?it/s]

  9%|▉         | 511/5500 [00:00<00:00, 5073.16it/s]

 20%|██        | 1100/5500 [00:00<00:00, 5528.49it/s]

 31%|███       | 1715/5500 [00:00<00:00, 5791.95it/s]

 42%|████▏     | 2295/5500 [00:00<00:00, 5745.64it/s]

...

 75%|███████▍  | 4121/5500 [00:00<00:00, 5742.57it/s]

 86%|████████▋ | 4757/5500 [00:00<00:00, 5928.40it/s]

 98%|█████████▊| 5366/5500 [00:00<00:00, 5965.23it/s]

100%|██████████| 5500/5500 [00:00<00:00, 5847.50it/s]

LF matrix shape: (5500, 10)
Coverage (at least one LF fires): 70.2%


In [5]:
# Section 4 — Fit LabelModel, predict, save results
label_model = LabelModel(cardinality=IMAGE_NUM_CLASSES, verbose=True)
label_model.fit(L_train=L, n_epochs=500, seed=IMAGE_SEED)

pred_labels = label_model.predict(L=L, tie_break_policy="abstain")
# -1 means all LFs abstained — treat as no prediction
covered_mask = pred_labels != -1
print(f"Final coverage: {covered_mask.mean():.1%}")

true_labels = manifest["label_idx"].to_numpy()
label_acc = (pred_labels[covered_mask] == true_labels[covered_mask]).mean()
label_f1 = f1_score(true_labels[covered_mask], pred_labels[covered_mask], average="macro", zero_division=0)
print(f"Label ACC (covered): {label_acc:.4f}  F1: {label_f1:.4f}")

out_df = manifest[["filepath", "true_label"]].copy()
out_df["predicted_label"] = [IMAGE_CLASS_NAMES[p] if p != -1 else "ABSTAIN" for p in pred_labels]
RESULTS_DIR.mkdir(exist_ok=True)
out_df.to_csv(RESULTS_DIR / "image_full_labels_weak_supervision.csv", index=False)

metrics = {"label_acc": float(label_acc), "label_f1": float(label_f1), "coverage": float(covered_mask.mean())}
with open(RESULTS_DIR / "image_metrics_weak_supervision.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)

INFO:root:Computing O...


INFO:root:Estimating \mu...


  0%|          | 0/500 [00:00<?, ?epoch/s]

INFO:root:[0 epochs]: TRAIN:[loss=0.030]


INFO:root:[10 epochs]: TRAIN:[loss=0.025]


...

INFO:root:[480 epochs]: TRAIN:[loss=0.002]


INFO:root:[490 epochs]: TRAIN:[loss=0.002]


100%|██████████| 500/500 [00:00<00:00, 637.47epoch/s]


INFO:root:Finished Training


Final coverage: 70.2%
Label ACC (covered): 0.9775  F1: 0.9783
{'label_acc': 0.9774786435412891, 'label_f1': 0.9782540373091411, 'coverage': 0.7023636363636364}
